In [1]:
# Config,
MODEL_ID = "google/gemma-4-E2B-it"
OUTPUT_DIR = "./gemma4-e2b-lora-cpu"
DATASET_PATH = "MonkeyIslandStylePirateDuelDataset.jsonl"
MAX_LENGTH = 256
NUM_EPOCHS = 10
LEARNING_RATE = 1e-3
SYSTEM_PROMPT = None

In [2]:
# Load model,
import torch
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float32,    # You may use bfloat16 if your cpu supports it. Try "!lscpu | grep bf16", if you don't see "avx512_bf16" in the output, then don't enable bf16, it's only going to slow down training.
    device_map=None,
    low_cpu_mem_usage=True,
)
model.config.use_cache = False   # Must be set to False during training, because KV caching is not needed. Only turn on during inference.

# Gradient checkpointing saves RAM,
model.gradient_checkpointing_enable()

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

In [3]:
# Load dataset,
import json

def load_dataset(path):
    dataset = []
    
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            dataset.append(json.loads(line))
    
    return dataset
    
print("Loading dataset...")
data = load_dataset(DATASET_PATH)
print(data[0])

Loading dataset...
{'prompt': '<action_insult>', 'completion': 'You fight like a dairy farmer.'}


In [4]:
# Format into Gemma 4 prompt format (See: https://ai.google.dev/gemma/docs/core/prompt-formatting-gemma4 ),
# Returns:
#     "prompt" is the prefix, the part that is not included in the loss calculation.
#     "completion" is the entire text.
def gemma4Format(prompt: str, completion: str, system_prompt: str | None = None) -> str:
    prompt = prompt.strip()
    prefix = ""
    if system_prompt and system_prompt.strip():
        system_prompt = system_prompt.strip()
        prefix = (
            "<|turn>system\n"
            f"{system_prompt}<turn|>\n")
        
    prefix = (
        f"{prefix}"
        "<|turn>user\n"
        f"{prompt}<turn|>\n"
        "<|turn>model\n")
    completion = (
        f"{prefix}"
        f"{completion}<turn|>\n")
    return {"prompt": prefix, "completion": completion}

formattedData = []
for entry in data:
    formattedEntry = gemma4Format(system_prompt=SYSTEM_PROMPT, prompt=entry["prompt"], completion=entry['completion'])
    formattedData.append(formattedEntry)

print(formattedData[0])

{'prompt': '<|turn>user\n<action_insult><turn|>\n<|turn>model\n', 'completion': '<|turn>user\n<action_insult><turn|>\n<|turn>model\nYou fight like a dairy farmer.<turn|>\n'}


In [5]:
# Tokenizer,
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_example(example: Dict[str, str]) -> Dict[str, List[int]]:
    prompt_ids = tokenizer(
        example["prompt"],
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LENGTH,
    )["input_ids"]

    full_encoded = tokenizer(
        example["completion"],
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LENGTH,
    )

    input_ids = full_encoded["input_ids"]
    attention_mask = full_encoded["attention_mask"]

    # Only the assistant completion and closing turn token contribute to loss.
    # Mask everything up to the start of the completion. The default magic 
    # number for PyTorch's loss functions is -100. We'll insert -100 for every 
    # token we want to exclude,
    prefix_len = min(len(prompt_ids), len(input_ids))  # input_ids might have been truncated too much that it's smaller than prompt_ids.
    labels = ([-100] * prefix_len) + input_ids[prefix_len:]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }


tokenized_dataset = []
for entry in formattedData:
    tokenizedEntry = tokenize_example({"prompt": entry["prompt"], "completion": entry["completion"]})
    tokenized_dataset.append(tokenizedEntry)

print(tokenized_dataset[0])

{'input_ids': [105, 2364, 107, 236820, 2064, 236779, 1365, 745, 236813, 106, 107, 105, 4368, 107, 3048, 6093, 1133, 496, 26283, 27155, 236761, 106, 107], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, 3048, 6093, 1133, 496, 26283, 27155, 236761, 106, 107]}


In [6]:
# Lora config,
# A PEFT model is a base neural network (like Llama or Mistral) that has been modified using Parameter-Efficient Fine-Tuning techniques to learn new tasks without changing most of its original parameters.
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=r".*\.language_model.*\.(q_proj|k_proj|v_proj|o_proj)"
    # target_modules=[
    #     "q_proj",
    #     "k_proj",
    #     "v_proj",
    #     "o_proj",
    #     #"gate_proj",
    #     #"up_proj",
    #     #"down_proj",        
    # ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 2,678,784 || all params: 5,106,976,288 || trainable%: 0.0525


In [7]:
from transformers import TrainingArguments

# !lscpu | grep bf16
# If you don't see "avx512_bf16" in the output, then don't enable bf16, it's only going to slow down training.

# Training arguments,
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    logging_steps=1,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=2,
    bf16=False,
    fp16=False,
    optim="adamw_torch",
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=0,
)

In [8]:
# Train,
from transformers import (
    Trainer,
    DataCollatorForLanguageModeling,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False,
    ),
)

print("Starting training...")
trainer.train()

print("Saving adapter...")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Done.")


Starting training...


/root/LoraTraining/lib/python3.14/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,7.766061
2,6.596803
3,4.737574
4,3.275507
5,2.772048
6,1.965451
7,1.726050
8,1.852030
9,1.583643
10,1.513938


/root/LoraTraining/lib/python3.14/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/root/LoraTraining/lib/python3.14/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/root/LoraTraining/lib/python3.14/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/root/LoraTraining/lib/python3.14/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Saving adapter...
Done.


In [9]:
# Clean up (you might want to restart kernel and continue from below to clean more memory),
for var in ["model", "tokenizer", "trainer", "dataset"]:
    if var in globals():
        del globals()[var]

import gc
gc.collect()

1088

In [2]:
# Test,
import torch
from transformers import AutoModelForCausalLM

print("\nReloading model for inference test...")

# Reload clean base model (we could have used the already loaded one, but we are also making sure
# that we saved the model properly),
test_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float32,
    low_cpu_mem_usage=True,
)


Reloading model for inference test...


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

In [3]:
# Attach LoRA adapter,
from peft import PeftModel
test_model = PeftModel.from_pretrained(
    test_model,
    OUTPUT_DIR,
)
test_model.eval()
test_model.config.use_cache = True
torch.set_num_threads(12)

# Load tokenizer,
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [234]:
# Create a test prompt,
def build_prompt(prompt: str, system_prompt: str | None = None) -> str:
    prompt = prompt.strip()
    if system_prompt and system_prompt.strip():
        system_prompt = system_prompt.strip()
        return (
            "<|turn>system\n"
            f"{system_prompt}<turn|>\n"
            "<|turn>user\n"
            f"{prompt}<turn|>\n"
            "<|turn>model\n"
        )
        
    return (
        "<|turn>user\n"
        f"{prompt}<turn|>\n"
        "<|turn>model\n"
    )

#test_prompt = "<action_insult>"
#test_prompt = "<action_reply><insult>Your beard looks like a shipwrecked mop.<reply>"
#test_prompt = "<action_reply><insult>You are ugly.<reply>"
#test_prompt = "<action_reply><insult>You fight like a rubber duck.<reply>"
#test_prompt = "<action_reply><insult>People run when they hear my name.<reply>"
test_prompt = "<action_reply><insult>I'll carve my name into your flesh.<reply>"
#test_prompt = "<action_judge><insult>I'll carve my name into your flesh.<reply>Then you've lost the fight to death!<grade>"
#test_prompt = "<action_judge><insult>People .<reply>...<grade>"
#test_prompt = "<action_judge><insult>People run when they hear my name.<reply>That's cute. I fight back.<grade>"
#test_prompt = "<action_judge><insult>People run when they hear my name.<reply>Even before they smell your stench?!<grade>"
#test_prompt = "<action_judge><insult>You fight like a rubber duck.<reply>Yet you’re still the one getting absolutely soaked.<grade>"
formatted_prompt = build_prompt(
    test_prompt,
    system_prompt=SYSTEM_PROMPT
)

inputs = tokenizer(
    formatted_prompt,
    return_tensors="pt",
)

# Generate, one word at a time,
print("\nGenerating...\n")
from transformers import TextIteratorStreamer
from threading import Thread

streamer = TextIteratorStreamer(
    tokenizer,
    skip_prompt=True,
    skip_special_tokens=True,
)

generation_kwargs = dict(
    **inputs,
    max_new_tokens=2048,
    do_sample=True,
    temperature=1.12, # 1.12 when replying. 2.24 when insulting.
    top_p=1,
    top_k=30,         # 1 when judging, 30 otherwise.
    repetition_penalty=1.3,
    pad_token_id=tokenizer.eos_token_id,
    streamer=streamer,
)

# Run generation in background thread
thread = Thread(
    target=test_model.generate,
    kwargs=generation_kwargs,
)

thread.start()

# Print tokens as they arrive
for text in streamer:
    print(text, end="", flush=True)

print("\n---")


Generating...

And you look like you're going to do it.
---


In [4]:
# Convert into gguf,
# See: https://github.com/ggml-org/llama.cpp/pull/23077/files
# Took convert_lora_to_gguf.py and convert_hf_to_gguf.py from there.

#!python3 /app/Models/llama.cpp/convert_lora_to_gguf.py gemma4-e2b-lora-cpu/ --base /app/Models/gemma-4-E2B-it
#!python3 /app/Models/llama.cpp/convert_hf_to_gguf.py /app/Models/TextAdventure/gemma4-merged --outfile gemma4-merged.gguf --outtype auto
#!./build/bin/llama-quantize /app/Models/TextAdventure/gemma4-merged.gguf /app/Models/TextAdventure/gemma4-merged-q4_k_m.gguf q4_k_m

#print("Saving model merged with LoRA...")
#test_model = test_model.merge_and_unload()
#test_model.save_pretrained("gemma4-merged")


Saving unified Hugging Face model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]